# Tabla con todo menos métricas intervencionales

In [1]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

NOTEBOOKS_DIR = Path.cwd().parent if Path.cwd().name == "Resumenes" else Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent

ALPHA = 0.05
METRICAS = ["MAE Z", "HSIC(Z,X)", "HSIC(Z,Y)", "RF Acc"]  # menor es mejor en las 4

## Definición de los 8 experimentos observacionales

In [2]:
EXPERIMENTOS = {
    "Aditivo Gaussiano": {"path": NOTEBOOKS_DIR / "Experimento1_aditivo/Observacional/tablas/aditivo_gausian.csv"},
    "Aditivo Exponencial": {"path": NOTEBOOKS_DIR / "Experimento1_aditivo/Observacional/tablas/aditivo_exponential.csv"},
    "Aditivo Gamma": {"path": NOTEBOOKS_DIR / "Experimento1_aditivo/Observacional/tablas/aditivo_gamma.csv"},
    "Aditivo Uniforme": {"path": NOTEBOOKS_DIR / "Experimento1_aditivo/Observacional/tablas/aditivo_uniform.csv"},
    "Multiplicativo Gaussiano": {"path": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Observacional/tablas/multiplicativo_gausian.csv"},
    "Multiplicativo Exponencial": {"path": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Observacional/tablas/multiplicativo_exponencial.csv"},
    "Multiplicativo Gamma": {"path": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Observacional/tablas/multiplicativo_gamma.csv"},
    "Multiplicativo Uniforme": {"path": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Observacional/tablas/multiplicativo_uniforme.csv"},
}

for nombre, info in EXPERIMENTOS.items():
    assert info["path"].exists(), f"No existe: {info['path']}"


## Función de selección de la Beta óptima

In [3]:
def beta_optima(df: pd.DataFrame, n_filter: int, metrics=METRICAS):
    """Selecciona la Beta óptima (Beta != 0) por prioridad estricta: RF Acc -> MMD -> MAE Z -> Beta."""
    df_n = df[df["N"] == n_filter]
    columnas = sorted(set(metrics) | {"MMD"})
    media_por_beta = df_n.groupby("Beta")[columnas].mean()

    baseline = media_por_beta.loc[0.0]
    media_no_cero = media_por_beta.drop(index=0.0)

    orden_desempate = ["RF Acc", "MMD", "MAE Z"]
    candidatas = media_no_cero[orden_desempate].reset_index().sort_values(
        by=orden_desempate + ["Beta"],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])
    valores_opt = media_por_beta.loc[beta_opt]
    return baseline, beta_opt, valores_opt


## Función de test de Wilcoxon pareado

In [4]:
def wilcoxon_experimento(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Wilcoxon signed-rank pareado (por Seed) entre Beta=0 y Beta=beta_opt, por métrica."""
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index("Seed")[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index("Seed")[list(metrics)]
    seeds_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[seeds_comunes]
    opt = opt.loc[seeds_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(seeds_comunes)


## Cálculo (tabla resumen + p-valores) para un N dado

In [5]:
def analizar_n(n_filter: int):
    """Ejecuta beta_optima + wilcoxon_experimento para los 8 experimentos, a un N fijo.

    Además de las 4 METRICAS, guarda MMD (ya calculada por beta_optima) y su propio p-valor,
    para poder construir más adelante una tabla RF/MMD observacional vs. intervencional.
    """
    filas_resumen = []
    filas_p = []

    for nombre, info in EXPERIMENTOS.items():
        df = pd.read_csv(info["path"])
        baseline, beta_opt, valores_opt = beta_optima(df, n_filter=n_filter)

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "MAE(Z) beta=0": baseline["MAE Z"],
            "MAE(Z) beta_optima": valores_opt["MAE Z"],
            "HSIC(Z,X) beta=0": baseline["HSIC(Z,X)"],
            "HSIC(Z,X) beta_optima": valores_opt["HSIC(Z,X)"],
            "HSIC(Z,Y) beta=0": baseline["HSIC(Z,Y)"],
            "HSIC(Z,Y) beta_optima": valores_opt["HSIC(Z,Y)"],
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "MMD beta=0": baseline["MMD"],
            "MMD beta_optima": valores_opt["MMD"],
        })

        p_valores, n_pares = wilcoxon_experimento(df, beta_opt, n_filter=n_filter)
        p_valores_mmd, _ = wilcoxon_experimento(df, beta_opt, n_filter=n_filter, metrics=["MMD"])
        p_valores["MMD"] = p_valores_mmd["MMD"]
        filas_p.append({"Experimento": nombre, **p_valores, "n_seeds_pareadas": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values


resumen_50, p_values_50 = analizar_n(50)
resumen_100, p_values_100 = analizar_n(100)


## Mapas de métricas y traducción de etiquetas (Distribution/Noise a inglés)

In [6]:
COLUMNA_A_METRICA = {
    "MAE(Z) beta_optima": "MAE Z",
    "HSIC(Z,X) beta_optima": "HSIC(Z,X)",
    "HSIC(Z,Y) beta_optima": "HSIC(Z,Y)",
    "RF Acc beta_optima": "RF Acc",
}
METRICA_A_COLUMNA = {v: k for k, v in COLUMNA_A_METRICA.items()}
METRICA_ORDER = ["RF Acc", "MAE Z", "HSIC(Z,Y)", "HSIC(Z,X)"]

NOISE_EN = {"Aditivo": "Additive", "Multiplicativo": "Non-additive"}
DISTRIBUTION_EN = {
    "Gaussiano": "Gaussian",
    "Exponencial": "Exponential",
    "Gamma": "Gamma",
    "Uniforme": "Uniform",
}
DISTRIBUTION_ORDER = ["Gaussiano", "Exponencial", "Gamma", "Uniforme"]
NOISE_ORDER = ["Aditivo", "Multiplicativo"]
NOISE_ES = {v: k for k, v in NOISE_EN.items()}
DISTRIBUTION_ES = {v: k for k, v in DISTRIBUTION_EN.items()}


## Tabla combinada (N=50 y N=100), sin métricas intervencionales

In [7]:
filas = []
for dist in DISTRIBUTION_ORDER:
    for noise in NOISE_ORDER:
        experimento = f"{noise} {dist}"
        for n_filter, resumen in [(50, resumen_50), (100, resumen_100)]:
            fila_resumen = resumen.set_index("Experimento").loc[experimento]
            fila = {
                "Distribution": DISTRIBUTION_EN[dist],
                "Noise": NOISE_EN[noise],
                "N": n_filter,
                "Beta": fila_resumen["beta_optima"],
            }
            for metrica in METRICA_ORDER:
                col_opt = METRICA_A_COLUMNA[metrica]
                col_mse = col_opt.replace("beta_optima", "beta=0")
                fila[f"{metrica} MSE"] = fila_resumen[col_mse]
                fila[f"{metrica} beta_opt"] = fila_resumen[col_opt]
            filas.append(fila)

combined_table = pd.DataFrame(filas)
metric_cols = [c for c in combined_table.columns if c not in ("Distribution", "Noise", "N", "Beta")]
combined_table[metric_cols] = combined_table[metric_cols].round(5)
combined_table["Beta"] = combined_table["Beta"].round(2)
combined_table = combined_table.set_index(["Distribution", "Noise", "N"])


## Resaltado de celdas significativas y visualización

In [8]:
def resaltar_significativas(row):
    dist_en, noise_en, n_filter = row.name
    experimento = f"{NOISE_ES[noise_en]} {DISTRIBUTION_ES[dist_en]}"
    p_values = p_values_50 if n_filter == 50 else p_values_100
    p_exp = p_values.loc[experimento]

    estilos = []
    for col in row.index:
        if col.endswith(" beta_opt"):
            metrica = col[: -len(" beta_opt")]
            if pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
                continue
        estilos.append("")
    return estilos


combined_table.style.apply(resaltar_significativas, axis=1).set_table_styles(
    [
        {"selector": "th.row_heading.level0", "props": [("text-align", "center")]},
        {"selector": "th.row_heading.level1", "props": [("text-align", "center")]},
    ],
    overwrite=False,
)


## Importación de datos intervencionales (igual que `Resumen_Intervencional.ipynb`)

Mismos 8 CSVs y mismas rutas que `Resumen_Intervencional.ipynb`. `METRICAS_INTERVENCIONAL` no incluye
`HSIC(Z,Y)`: bajo intervención, `Y` se fija externamente (`do(Y)`) y ya no se genera por su ecuación
estructural, así que esa columna viene vacía en estos CSVs. Cada CSV tiene además una columna `Y_do`
(`0.0` / `1.0`) que no existe en los datos observacionales.

In [9]:
METRICAS_INTERVENCIONAL = ["MAE Z", "HSIC(Z,X)", "RF Acc"]  # sin HSIC(Z,Y): no aplica bajo do(Y)

EXPERIMENTOS_INTERVENCIONAL = {
    "Aditivo Gaussiano": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_gausian.csv",
    "Aditivo Exponencial": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_exponencial.csv",
    "Aditivo Gamma": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_gamma.csv",
    "Aditivo Uniforme": NOTEBOOKS_DIR / "Experimento1_aditivo/Intervencional-20/tablas/intervencional_aditivo_uniforme.csv",
    "Multiplicativo Gaussiano": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_gausian.csv",
    "Multiplicativo Exponencial": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_exponencial.csv",
    "Multiplicativo Gamma": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_gamma.csv",
    "Multiplicativo Uniforme": NOTEBOOKS_DIR / "Experimento1_no_aditivo/Intervencional/tablas/intervencional_multiplicativo_uniforme.csv",
}

for nombre, path in EXPERIMENTOS_INTERVENCIONAL.items():
    assert path.exists(), f"No existe: {path}"


In [10]:
datos_intervencionales = {
    nombre: pd.read_csv(path) for nombre, path in EXPERIMENTOS_INTERVENCIONAL.items()
}


## Beta óptima observacional (para aplicar a los datos intervencionales)

Igual que en `Resumen_Intervencional.ipynb`: la Beta usada aquí no se re-selecciona a partir de las
métricas intervencionales, se toma la ya elegida por `beta_optima` sobre los datos observacionales
(`resumen_50`/`resumen_100`, calculados arriba). Solo cambia entre N=50 y N=100.

In [11]:
BETA_OPTIMA_OBSERVACIONAL = {}
for n_filter, resumen in [(50, resumen_50), (100, resumen_100)]:
    for _, fila in resumen.iterrows():
        BETA_OPTIMA_OBSERVACIONAL[(fila["Experimento"], n_filter)] = float(fila["beta_optima"])


## Funciones de RF int / MMD int (media entre do(Y)=0.0 y do(Y)=1.0)

`RF int` y `MMD int` son las versiones intervencionales de `RF Acc` y `MMD`: se toman de los mismos
CSVs que `Resumen_Intervencional.ipynb`, pero solo se usa la versión "media" (no se separa por
`do(Y)`), así que no hace falta filtrar por `Y_do`: promediar por `Beta` ya mezcla ambas
intervenciones. El test de Wilcoxon sí se empareja por `(Seed, Y_do)`, igual que en la versión
"media" de `Resumen_Intervencional.ipynb`.

In [12]:
METRICAS_INT = ["RF Acc", "MMD"]


def valores_para_beta_int(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS_INT):
    """Baseline (Beta=0) y valores en Beta=beta_opt, promediando por seed y por Y_do (media)."""
    df_n = df[df["N"] == n_filter]
    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()
    baseline = media_por_beta.loc[0.0]
    valores_opt = media_por_beta.loc[beta_opt]
    return baseline, valores_opt


def wilcoxon_experimento_int(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS_INT):
    """Wilcoxon signed-rank pareado por (Seed, Y_do) entre Beta=0 y Beta=beta_opt."""
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index(["Seed", "Y_do"])[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index(["Seed", "Y_do"])[list(metrics)]
    pares_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[pares_comunes]
    opt = opt.loc[pares_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(pares_comunes)


## Cálculo (RF int + MMD int, media do(Y)) para un N dado

In [13]:
def analizar_n_int(n_filter: int):
    """RF Acc y MMD intervencionales (media do(Y)=0.0/1.0), usando la Beta óptima observacional."""
    filas_resumen = []
    filas_p = []

    for nombre, df in datos_intervencionales.items():
        beta_opt = BETA_OPTIMA_OBSERVACIONAL[(nombre, n_filter)]
        baseline, valores_opt = valores_para_beta_int(df, beta_opt, n_filter)

        filas_resumen.append({
            "Experimento": nombre,
            "beta_optima": beta_opt,
            "RF Acc beta=0": baseline["RF Acc"],
            "RF Acc beta_optima": valores_opt["RF Acc"],
            "MMD beta=0": baseline["MMD"],
            "MMD beta_optima": valores_opt["MMD"],
        })

        p_valores, n_pares = wilcoxon_experimento_int(df, beta_opt, n_filter)
        filas_p.append({"Experimento": nombre, **p_valores, "n_pares": n_pares})

    resumen = pd.DataFrame(filas_resumen)
    p_values = pd.DataFrame(filas_p).set_index("Experimento")
    return resumen, p_values


resumen_int_50, p_values_int_50 = analizar_n_int(50)
resumen_int_100, p_values_int_100 = analizar_n_int(100)


## Tabla combinada intervencional (RF int, MMD int; media do(Y); N=50 y N=100)

In [14]:
METRICA_A_COLUMNA_INT = {
    "RF Acc": "RF Acc beta_optima",
    "MMD": "MMD beta_optima",
}
DISPLAY_METRICA_INT = {"RF Acc": "RF int", "MMD": "MMD int"}
METRICA_DESDE_ETIQUETA_INT = {v: k for k, v in DISPLAY_METRICA_INT.items()}
METRICA_ORDER_INT = ["RF Acc", "MMD"]

filas = []
for dist in DISTRIBUTION_ORDER:
    for noise in NOISE_ORDER:
        experimento = f"{noise} {dist}"
        for n_filter, resumen in [(50, resumen_int_50), (100, resumen_int_100)]:
            fila_resumen = resumen.set_index("Experimento").loc[experimento]
            fila = {
                "Distribution": DISTRIBUTION_EN[dist],
                "Noise": NOISE_EN[noise],
                "N": n_filter,
                "Beta": fila_resumen["beta_optima"],
            }
            for metrica in METRICA_ORDER_INT:
                col_opt = METRICA_A_COLUMNA_INT[metrica]
                col_mse = col_opt.replace("beta_optima", "beta=0")
                etiqueta = DISPLAY_METRICA_INT[metrica]
                fila[f"{etiqueta} MSE"] = fila_resumen[col_mse]
                fila[f"{etiqueta} beta_opt"] = fila_resumen[col_opt]
            filas.append(fila)

combined_table_int = pd.DataFrame(filas)
metric_cols_int = [c for c in combined_table_int.columns if c not in ("Distribution", "Noise", "N", "Beta")]
combined_table_int[metric_cols_int] = combined_table_int[metric_cols_int].round(5)
combined_table_int["Beta"] = combined_table_int["Beta"].round(2)
combined_table_int = combined_table_int.set_index(["Distribution", "Noise", "N"])


## Resaltado de celdas significativas y visualización (RF int / MMD int)

In [15]:
def resaltar_significativas_int(row):
    dist_en, noise_en, n_filter = row.name
    experimento = f"{NOISE_ES[noise_en]} {DISTRIBUTION_ES[dist_en]}"
    p_values = p_values_int_50 if n_filter == 50 else p_values_int_100
    p_exp = p_values.loc[experimento]

    estilos = []
    for col in row.index:
        if col.endswith(" beta_opt"):
            etiqueta = col[: -len(" beta_opt")]
            metrica = METRICA_DESDE_ETIQUETA_INT[etiqueta]
            if pd.notna(p_exp[metrica]) and p_exp[metrica] < ALPHA:
                estilos.append("font-weight: bold")
                continue
        estilos.append("")
    return estilos


combined_table_int.style.apply(resaltar_significativas_int, axis=1).set_table_styles(
    [
        {"selector": "th.row_heading.level0", "props": [("text-align", "center")]},
        {"selector": "th.row_heading.level1", "props": [("text-align", "center")]},
    ],
    overwrite=False,
)


## Tabla conjunta (observacional + intervencional)

Une `combined_table` y `combined_table_int` por su índice común (Distribution, Noise, N): las
métricas intervencionales (`RF int`, `MMD int`) se añaden a la derecha de las observacionales. `Beta`
es la misma en ambas tablas (la interventional reutiliza la Beta óptima observacional), así que solo
se mantiene una vez.

In [16]:
combined_full = combined_table.join(combined_table_int.drop(columns="Beta"))

obs_cols = [c for c in combined_table.columns if c != "Beta"]
int_cols = [c for c in combined_table_int.columns if c != "Beta"]

combined_full.style.apply(resaltar_significativas, axis=1, subset=obs_cols).apply(
    resaltar_significativas_int, axis=1, subset=int_cols
).set_table_styles(
    [
        {"selector": "th.row_heading.level0", "props": [("text-align", "center")]},
        {"selector": "th.row_heading.level1", "props": [("text-align", "center")]},
    ],
    overwrite=False,
)


## Tabla RF / MMD (observacional vs. intervencional)

Subconjunto de la tabla conjunta: solo `RF obs`, `MMD obs`, `RF int` y `MMD int`. `MMD obs` no estaba
en `combined_table` (esta solo tenía `RF Acc`, `MAE Z`, `HSIC(Z,Y)`, `HSIC(Z,X)`), así que se toma de
`resumen_50`/`resumen_100`, que ya calculan `MMD` (columna auxiliar interna de `beta_optima`) y su
propio p-valor de Wilcoxon.

In [17]:
filas = []
for dist in DISTRIBUTION_ORDER:
    for noise in NOISE_ORDER:
        experimento = f"{noise} {dist}"
        for n_filter, resumen_obs, resumen_int in [
            (50, resumen_50, resumen_int_50),
            (100, resumen_100, resumen_int_100),
        ]:
            fila_obs = resumen_obs.set_index("Experimento").loc[experimento]
            fila_int = resumen_int.set_index("Experimento").loc[experimento]
            filas.append({
                "Distribution": DISTRIBUTION_EN[dist],
                "Noise": NOISE_EN[noise],
                "N": n_filter,
                "Beta": fila_obs["beta_optima"],
                "RF obs MSE": fila_obs["RF Acc beta=0"],
                "RF obs beta_opt": fila_obs["RF Acc beta_optima"],
                "MMD obs MSE": fila_obs["MMD beta=0"],
                "MMD obs beta_opt": fila_obs["MMD beta_optima"],
                "RF int MSE": fila_int["RF Acc beta=0"],
                "RF int beta_opt": fila_int["RF Acc beta_optima"],
                "MMD int MSE": fila_int["MMD beta=0"],
                "MMD int beta_opt": fila_int["MMD beta_optima"],
            })

rf_mmd_table = pd.DataFrame(filas)
metric_cols_rf_mmd = [c for c in rf_mmd_table.columns if c not in ("Distribution", "Noise", "N", "Beta")]
rf_mmd_table[metric_cols_rf_mmd] = rf_mmd_table[metric_cols_rf_mmd].round(5)
rf_mmd_table["Beta"] = rf_mmd_table["Beta"].round(2)
rf_mmd_table = rf_mmd_table.set_index(["Distribution", "Noise", "N"])


## Resaltado de celdas significativas y visualización (RF obs / MMD obs / RF int / MMD int)

In [18]:
METRICA_DESDE_ETIQUETA_OBS = {"RF obs": "RF Acc", "MMD obs": "MMD"}


def resaltar_significativas_rf_mmd(row):
    dist_en, noise_en, n_filter = row.name
    experimento = f"{NOISE_ES[noise_en]} {DISTRIBUTION_ES[dist_en]}"
    p_obs = p_values_50 if n_filter == 50 else p_values_100
    p_int = p_values_int_50 if n_filter == 50 else p_values_int_100
    p_exp_obs = p_obs.loc[experimento]
    p_exp_int = p_int.loc[experimento]

    estilos = []
    for col in row.index:
        if col.endswith(" beta_opt"):
            etiqueta = col[: -len(" beta_opt")]
            if etiqueta in METRICA_DESDE_ETIQUETA_OBS:
                p_val = p_exp_obs[METRICA_DESDE_ETIQUETA_OBS[etiqueta]]
            else:
                p_val = p_exp_int[METRICA_DESDE_ETIQUETA_INT[etiqueta]]
            if pd.notna(p_val) and p_val < ALPHA:
                estilos.append("font-weight: bold")
                continue
        estilos.append("")
    return estilos


rf_mmd_table.style.apply(resaltar_significativas_rf_mmd, axis=1).set_table_styles(
    [
        {"selector": "th.row_heading.level0", "props": [("text-align", "center")]},
        {"selector": "th.row_heading.level1", "props": [("text-align", "center")]},
    ],
    overwrite=False,
)


## Tabla pequeña: ruido Gamma (Additive y Non-additive), N=100, con p-valor por columna beta_opt

Mismas columnas que `rf_mmd_table`, pero filtrada a Gamma/N=100 y con un `<metrica> p-value` explícito
junto a cada `<metrica> beta_opt` (en vez de solo negrita).

In [19]:
filas = []
for noise in NOISE_ORDER:
    experimento = f"{noise} Gamma"
    n_filter = 50
    fila_obs = resumen_50.set_index("Experimento").loc[experimento]
    fila_int = resumen_int_50.set_index("Experimento").loc[experimento]
    p_exp_obs = p_values_50.loc[experimento]
    p_exp_int = p_values_int_50.loc[experimento]
    filas.append({
        "Distribution": DISTRIBUTION_EN["Gamma"],
        "Noise": NOISE_EN[noise],
        "Beta": fila_obs["beta_optima"],
        "RF obs MSE": fila_obs["RF Acc beta=0"],
        "RF obs beta_opt": fila_obs["RF Acc beta_optima"],
        "RF obs p-value": p_exp_obs["RF Acc"],
        "MMD obs MSE": fila_obs["MMD beta=0"],
        "MMD obs beta_opt": fila_obs["MMD beta_optima"],
        "MMD obs p-value": p_exp_obs["MMD"],
        "RF int MSE": fila_int["RF Acc beta=0"],
        "RF int beta_opt": fila_int["RF Acc beta_optima"],
        "RF int p-value": p_exp_int["RF Acc"],
        "MMD int MSE": fila_int["MMD beta=0"],
        "MMD int beta_opt": fila_int["MMD beta_optima"],
        "MMD int p-value": p_exp_int["MMD"],
    })

gamma_n50_table = pd.DataFrame(filas)
metric_cols_gamma = [
    c for c in gamma_n50_table.columns
    if c not in ("Distribution", "Noise", "Beta") and not c.endswith("p-value")
]
pvalue_cols_gamma = [c for c in gamma_n50_table.columns if c.endswith("p-value")]
gamma_n50_table[metric_cols_gamma] = gamma_n50_table[metric_cols_gamma].round(5)
gamma_n50_table[pvalue_cols_gamma] = gamma_n50_table[pvalue_cols_gamma].round(4)
gamma_n50_table["Beta"] = gamma_n50_table["Beta"].round(2)
gamma_n50_table = gamma_n50_table.set_index(["Distribution", "Noise"])


def resaltar_significativas_gamma_n50(row):
    dist_en, noise_en = row.name
    experimento = f"{NOISE_ES[noise_en]} {DISTRIBUTION_ES[dist_en]}"
    p_exp_obs = p_values_50.loc[experimento]
    p_exp_int = p_values_int_50.loc[experimento]

    estilos = []
    for col in row.index:
        if col.endswith(" beta_opt"):
            etiqueta = col[: -len(" beta_opt")]
            if etiqueta in METRICA_DESDE_ETIQUETA_OBS:
                p_val = p_exp_obs[METRICA_DESDE_ETIQUETA_OBS[etiqueta]]
            else:
                p_val = p_exp_int[METRICA_DESDE_ETIQUETA_INT[etiqueta]]
            if pd.notna(p_val) and p_val < ALPHA:
                estilos.append("font-weight: bold")
                continue
        estilos.append("")
    return estilos


gamma_n50_table.style.apply(resaltar_significativas_gamma_n50, axis=1).set_table_styles(
    [{"selector": "th.row_heading.level0", "props": [("text-align", "center")]}],
    overwrite=False,
)


## Tabla pequeña: solo ruido Non-additive (Gamma, N=50)

`gamma_n50_table` filtrada a `Noise == "Non-additive"`, con la columna `Noise` eliminada (`.xs`,
`level="Noise"`) ya que queda constante.

In [20]:
gamma_n50_nonadditive_table = gamma_n50_table.xs("Non-additive", level="Noise")


def resaltar_significativas_gamma_n50_nonadditive(row):
    dist_en = row.name
    experimento = f"{NOISE_ES['Non-additive']} {DISTRIBUTION_ES[dist_en]}"
    p_exp_obs = p_values_50.loc[experimento]
    p_exp_int = p_values_int_50.loc[experimento]

    estilos = []
    for col in row.index:
        if col.endswith(" beta_opt"):
            etiqueta = col[: -len(" beta_opt")]
            if etiqueta in METRICA_DESDE_ETIQUETA_OBS:
                p_val = p_exp_obs[METRICA_DESDE_ETIQUETA_OBS[etiqueta]]
            else:
                p_val = p_exp_int[METRICA_DESDE_ETIQUETA_INT[etiqueta]]
            if pd.notna(p_val) and p_val < ALPHA:
                estilos.append("font-weight: bold")
                continue
        estilos.append("")
    return estilos


gamma_n50_nonadditive_table.style.apply(resaltar_significativas_gamma_n50_nonadditive, axis=1).set_table_styles(
    [{"selector": "th.row_heading.level0", "props": [("text-align", "center")]}],
    overwrite=False,
)


,Beta,RF obs MSE,RF obs beta_opt,RF obs p-value,MMD obs MSE,MMD obs beta_opt,MMD obs p-value,RF int MSE,RF int beta_opt,RF int p-value,MMD int MSE,MMD int beta_opt,MMD int p-value
Distribution,,,,,,,,,,,,,
Gamma,0.600000,0.650000,0.618750,0.029800,0.056090,0.038360,0.008600,0.728750,0.688750,0.009700,0.077690,0.054340,0.057000


## Tabla RF / MMD con desviación estándar (subíndice)

Misma tabla que `rf_mmd_table` (RF obs / MMD obs / RF int / MMD int, MSE y beta_opt, N=50 y N=100),
pero cada media se muestra junto a su desviación estándar entre seeds como subíndice HTML
(`media<sub>±std</sub>`). La std observacional se recalcula desde los CSV crudos (`EXPERIMENTOS`,
cacheados una vez por experimento) y la std intervencional desde `datos_intervencionales`, agrupando
por `Beta` igual que se hizo para las medias (para la intervencional, sin filtrar `Y_do`, así que
mezcla ambas intervenciones tal como hace la versión "media"). `Beta` no lleva std: es un valor
elegido, no un promedio.

In [21]:
def std_por_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=("RF Acc", "MMD")):
    """Std entre seeds en Beta=0 y Beta=beta_opt, agrupando igual que las medias correspondientes."""
    df_n = df[df["N"] == n_filter]
    std_por_beta_tabla = df_n.groupby("Beta")[list(metrics)].std()
    return std_por_beta_tabla.loc[0.0], std_por_beta_tabla.loc[beta_opt]


def celda_con_std(media: float, std: float) -> str:
    return f"{media:.2f}<sub>{std:.2f}</sub>"


raw_obs = {nombre: pd.read_csv(info["path"]) for nombre, info in EXPERIMENTOS.items()}

filas = []
for dist in DISTRIBUTION_ORDER:
    for noise in NOISE_ORDER:
        experimento = f"{noise} {dist}"
        for n_filter, resumen_obs, resumen_int in [
            (50, resumen_50, resumen_int_50),
            (100, resumen_100, resumen_int_100),
        ]:
            beta_opt = BETA_OPTIMA_OBSERVACIONAL[(experimento, n_filter)]

            std_obs_mse, std_obs_opt = std_por_beta(raw_obs[experimento], beta_opt, n_filter)
            std_int_mse, std_int_opt = std_por_beta(datos_intervencionales[experimento], beta_opt, n_filter)

            fila_obs = resumen_obs.set_index("Experimento").loc[experimento]
            fila_int = resumen_int.set_index("Experimento").loc[experimento]

            filas.append({
                "Distribution": DISTRIBUTION_EN[dist],
                "Noise": NOISE_EN[noise],
                "N": n_filter,
                "Beta": round(beta_opt, 1),
                "RF obs MSE": celda_con_std(fila_obs["RF Acc beta=0"], std_obs_mse["RF Acc"]),
                "RF obs beta_opt": celda_con_std(fila_obs["RF Acc beta_optima"], std_obs_opt["RF Acc"]),
                "MMD obs MSE": celda_con_std(fila_obs["MMD beta=0"], std_obs_mse["MMD"]),
                "MMD obs beta_opt": celda_con_std(fila_obs["MMD beta_optima"], std_obs_opt["MMD"]),
                "RF int MSE": celda_con_std(fila_int["RF Acc beta=0"], std_int_mse["RF Acc"]),
                "RF int beta_opt": celda_con_std(fila_int["RF Acc beta_optima"], std_int_opt["RF Acc"]),
                "MMD int MSE": celda_con_std(fila_int["MMD beta=0"], std_int_mse["MMD"]),
                "MMD int beta_opt": celda_con_std(fila_int["MMD beta_optima"], std_int_opt["MMD"]),
            })

rf_mmd_table_std = pd.DataFrame(filas).set_index(["Distribution", "Noise", "N"])

rf_mmd_table_std.style.apply(resaltar_significativas_rf_mmd, axis=1).set_table_styles(
    [
        {"selector": "th.row_heading.level0", "props": [("text-align", "center")]},
        {"selector": "th.row_heading.level1", "props": [("text-align", "center")]},
    ],
    overwrite=False,
)
